# Skin Lesion Specialist -- ISIC 2019 Classifier (CliniqAI)

**Dataset:** [ISIC 2019 -- Skin Lesion Images for Melanoma Classification](https://www.kaggle.com/datasets/andrewmvd/isic-2019) (Kaggle mirror of the official ISIC 2019 challenge set)

- 25,331 dermoscopic images
- 8 diagnostic classes: MEL (melanoma), NV (melanocytic nevus), BCC (basal cell carcinoma),
  AK (actinic keratosis), BKL (benign keratosis), DF (dermatofibroma), VASC (vascular lesion),
  SCC (squamous cell carcinoma)
- Heavily imbalanced: NV and MEL alone make up the large majority of images, several classes
  (DF, VASC, SCC) have only a few hundred examples -- handled via class-weighted loss

**Split strategy:** by `lesion_id`, not by image. ISIC lesions are frequently photographed more
than once (different angle/zoom/time), so a random image-level split leaks the same lesion across
train/val/test. We split unique lesion IDs 70/15/15, then assign every image of a lesion to
whichever split its lesion landed in.

**Config (matches your other specialists):** EfficientNet-B0, batch size 32, lr 1e-3, weight
decay 1e-4, early stopping patience 10, 224x224 input, `num_workers=0` (Windows).


## 1. Session setup

Run this first in every session -- sets working directory and import path so the rest of your `CliniqAI` modules resolve correctly.

In [3]:
import os
import sys

# Adjust if your repo lives elsewhere
PROJECT_ROOT = r"E:\programming\CliniqAI"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", os.getcwd())

Working directory: E:\programming\CliniqAI


## 2. Download the dataset

Uses your existing Kaggle CLI config at `C:\Users\anish\.kaggle\kaggle.json`. The zip is
~9 GB (full-resolution dermoscopic images) -- make sure you have disk space before running this cell.

This only needs to run once. If `data/raw/skin_lesion` already has content, skip this cell.

In [4]:
import subprocess
from pathlib import Path

DATA_DIR = Path("data/raw/skin_lesion")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not any(DATA_DIR.iterdir()):
    subprocess.run([
        "kaggle", "datasets", "download",
        "-d", "andrewmvd/isic-2019",
        "-p", str(DATA_DIR),
        "--unzip"
    ], check=True)
    print("Download complete.")
else:
    print("Data already present, skipping download.")

Data already present, skipping download.


## 3. Locate images, ground-truth CSV, and metadata CSV

The Kaggle mirror typically unpacks to something like:

```
data/raw/skin_lesion/
    ISIC_2019_Training_Input/
        ISIC_0000000.jpg
        ...
    ISIC_2019_Training_GroundTruth.csv   # one-hot columns: image, MEL, NV, BCC, AK, BKL, DF, VASC, SCC
    ISIC_2019_Training_Metadata.csv      # image, lesion_id, age_approx, sex, anatom_site_general
```

This cell searches for those files regardless of exact nesting, since Kaggle mirrors vary slightly.

In [5]:
from pathlib import Path

DATA_DIR = Path("data/raw/skin_lesion")

def find_file(pattern):
    matches = list(DATA_DIR.rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"Could not find a file matching {pattern} under {DATA_DIR}")
    return matches[0]

def find_image_dir():
    candidates = [p for p in DATA_DIR.rglob("*") if p.is_dir() and any(p.glob("*.jpg"))]
    if not candidates:
        raise FileNotFoundError("Could not find a directory containing .jpg images")
    # Prefer the directory with the most images (avoids picking a small preview folder)
    return max(candidates, key=lambda d: len(list(d.glob("*.jpg"))))

GROUNDTRUTH_CSV = find_file("*GroundTruth*.csv")
METADATA_CSV = find_file("*Metadata*.csv")
IMAGE_DIR = find_image_dir()

print("Ground truth CSV:", GROUNDTRUTH_CSV)
print("Metadata CSV:", METADATA_CSV)
print("Image dir:", IMAGE_DIR, "-- image count:", len(list(IMAGE_DIR.glob("*.jpg"))))

Ground truth CSV: data\raw\skin_lesion\ISIC_2019_Training_GroundTruth.csv
Metadata CSV: data\raw\skin_lesion\ISIC_2019_Training_Metadata.csv
Image dir: data\raw\skin_lesion\ISIC_2019_Training_Input\ISIC_2019_Training_Input -- image count: 25331


## 4. Build per-image dataframe (label + lesion_id)

In [6]:
import pandas as pd

gt = pd.read_csv(GROUNDTRUTH_CSV)
meta = pd.read_csv(METADATA_CSV)

CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
present_classes = [c for c in CLASS_NAMES if c in gt.columns]
missing = set(CLASS_NAMES) - set(present_classes)
if missing:
    print(f"Note: classes not present in this CSV (older ISIC-2018 style has 7): {missing}")
CLASS_NAMES = present_classes

# One-hot -> single label
gt["label"] = gt[CLASS_NAMES].idxmax(axis=1)

df = gt[["image", "label"]].merge(meta[["image", "lesion_id"]], on="image", how="left")

# Some images have no recorded lesion_id (unique lesion, never re-photographed) -- treat each as its own lesion
df["lesion_id"] = df["lesion_id"].fillna(df["image"])

df["filepath"] = df["image"].apply(lambda x: str(IMAGE_DIR / f"{x}.jpg"))

print(f"Total images: {len(df)}")
print(f"Unique lesions: {df['lesion_id'].nunique()}")
print()
print(df["label"].value_counts())

C:\Users\anish\AppData\Local\Temp\ipykernel_22956\2180633232.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Total images: 25331
Unique lesions: 13931

label
NV      12875
MEL      4522
BCC      3323
BKL      2624
AK        867
SCC       628
VASC      253
DF        239
Name: count, dtype: int64


## 5. Lesion-level train/val/test split (70/15/15)

In [7]:
import random

random.seed(42)
lesion_ids = df["lesion_id"].unique().tolist()
random.shuffle(lesion_ids)

n = len(lesion_ids)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_lesions = set(lesion_ids[:n_train])
val_lesions = set(lesion_ids[n_train:n_train + n_val])
test_lesions = set(lesion_ids[n_train + n_val:])

train_df = df[df["lesion_id"].isin(train_lesions)].reset_index(drop=True)
val_df = df[df["lesion_id"].isin(val_lesions)].reset_index(drop=True)
test_df = df[df["lesion_id"].isin(test_lesions)].reset_index(drop=True)

print(f"Lesions -> train: {len(train_lesions)}, val: {len(val_lesions)}, test: {len(test_lesions)}")
print(f"Images  -> train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")
print()
print("Train class distribution:")
print(train_df["label"].value_counts())

Lesions -> train: 9751, val: 2089, test: 2091
Images  -> train: 17629, val: 3850, test: 3852

Train class distribution:
label
NV      8994
MEL     3136
BCC     2321
BKL     1889
AK       529
SCC      429
DF       170
VASC     161
Name: count, dtype: int64


## 6. Label encoding, dataset, transforms, dataloaders

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMG_SIZE = 224
BATCH_SIZE = 32

label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
idx_to_label = {i: name for name, i in label_to_idx.items()}

class ISICDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.paths = dataframe["filepath"].tolist()
        self.labels = [label_to_idx[l] for l in dataframe["label"].tolist()]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = ISICDataset(train_df, transform=train_transform)
val_dataset = ISICDataset(val_df, transform=eval_transform)
test_dataset = ISICDataset(test_df, transform=eval_transform)

# num_workers=0 required on Windows
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Batches -> train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")
print("Class index mapping:", idx_to_label)

Batches -> train: 551, val: 121, test: 121
Class index mapping: {0: 'MEL', 1: 'NV', 2: 'BCC', 3: 'AK', 4: 'BKL', 5: 'DF', 6: 'VASC', 7: 'SCC'}


## 7. Model -- EfficientNet-B0 (8-class head)

In [9]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

NUM_CLASSES = len(CLASS_NAMES)

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, NUM_CLASSES)
model = model.to(device)

print(model.classifier)

Device: cuda
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=8, bias=True)
)


## 8. Loss, optimizer, class weighting

ISIC 2019 is heavily imbalanced (NV and MEL dominate, DF/VASC/SCC are rare). Inverse-frequency
class weights keep rare classes from being ignored.

In [10]:
import numpy as np

class_counts = train_df["label"].value_counts().reindex(CLASS_NAMES).fillna(0)
total = class_counts.sum()
weights = total / (NUM_CLASSES * class_counts)
class_weights = torch.tensor(weights.values, dtype=torch.float32).to(device)

for name, w in zip(CLASS_NAMES, weights.values):
    print(f"  {name:>4}: count={int(class_counts[name]):>6}  weight={w:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

   MEL: count=  3136  weight=0.703
    NV: count=  8994  weight=0.245
   BCC: count=  2321  weight=0.949
    AK: count=   529  weight=4.166
   BKL: count=  1889  weight=1.167
    DF: count=   170  weight=12.963
  VASC: count=   161  weight=13.687
   SCC: count=   429  weight=5.137


## 9. Training loop with early stopping (patience=10)

In [ ]:
import time
from pathlib import Path

CHECKPOINT_DIR = Path("models/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "skin_lesion_efficientnet_b0.pt"

NUM_EPOCHS = 30
PATIENCE = 10

best_val_loss = float("inf")
epochs_no_improve = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def run_epoch(loader, train_mode):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    torch.set_grad_enabled(train_mode)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if train_mode:
            optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        if train_mode:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)
    elapsed = time.time() - start

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
          f"{elapsed:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
            "epoch": epoch,
            "class_names": CLASS_NAMES,
        }, CHECKPOINT_PATH)
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered after {epoch} epochs (no improvement for {PATIENCE} epochs).")
            break

print("\nTraining complete. Best val_loss:", best_val_loss)

Epoch 01/30 | train_loss=1.5796 train_acc=0.4555 | val_loss=1.3369 val_acc=0.5382 | 1132.2s
  -> saved new best checkpoint (val_loss=1.3369)
Epoch 02/30 | train_loss=1.3038 train_acc=0.5576 | val_loss=1.2780 val_acc=0.5899 | 835.6s
  -> saved new best checkpoint (val_loss=1.2780)
Epoch 03/30 | train_loss=1.1939 train_acc=0.5854 | val_loss=1.1788 val_acc=0.5948 | 854.2s
  -> saved new best checkpoint (val_loss=1.1788)
Epoch 04/30 | train_loss=1.0744 train_acc=0.5969 | val_loss=1.3835 val_acc=0.6049 | 933.5s
Epoch 05/30 | train_loss=1.0414 train_acc=0.6082 | val_loss=1.1173 val_acc=0.6260 | 820.5s
  -> saved new best checkpoint (val_loss=1.1173)


## 10. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 11. Load best checkpoint and evaluate on test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Test accuracy:", (np.array(all_preds) == np.array(all_labels)).mean())
print()
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Skin Lesion (ISIC 2019) -- Confusion Matrix")
plt.show()

## Notes

- **Checkpoint saved to:** `models/checkpoints/skin_lesion_efficientnet_b0.pt` -- sync to
  Google Drive (`medical-ai-models/trained_models/`) same as your other specialists.
- **Lesion-level split** (not image-level) means val/test accuracy is a more honest estimate of
  real-world performance -- multiple photos of the same lesion never span train/val/test.
- **8-way classification with heavy imbalance** is a genuinely harder task than your binary
  specialists (breast histology) or your visually-distinct-modality specialists (retinal/kidney/brain).
  Expect a lower headline accuracy here; look at per-class recall in the classification report,
  especially for MEL (melanoma) -- that's the clinically critical class to get right, even at the
  cost of some false positives on benign classes.
- If VRAM runs out at batch size 32 on the RTX 4050 (6GB), drop to 16.
- This slots in as a new Stage 2 modality class (`skin_lesion`) once you're ready to retrain the
  router's Stage 2 classifier alongside Roneet's modalities.
